# 12 - Spark

Paste each Bash block into an Ubuntu terminal in order. Keep one terminal for this notebook. Keep HDFS and the Python workbench running. This starts standalone Spark and submits a real Python job with a distributed executor. YARN can remain paused.

## Open your workspace

```bash
export WORKSPACE_HOME="${WORKSPACE_HOME:-$HOME/workspace}"
export PATH="$HOME/.local/bin:$PATH"
cd "$WORKSPACE_HOME"
kubectl config use-context bigdata
kubectl config set-context bigdata --namespace=bigdata
mkdir -p .state
```



Services: spark-master:7077 (submission), spark-master:8080 and spark-worker:8081 (web/native metrics). These daemons use emptyDir /data rather than PVCs; persistent job results go to HDFS or S3. Source is stored in pyspark/. Jupyter runs the driver. Driver Log4j logs are forwarded from /tmp/spark-logs; worker executor stdout/stderr are forwarded from /data/spark-work. Both Prometheus and Loki must be running.

## Ensure the workbench

```bash
bash start-components.sh jupyter
```



## Apply Spark

```bash
kubectl apply -f monitoring/telemetry-config.yaml
kubectl apply -f spark/spark.yaml
```



## Wait for spark-master

```bash
kubectl rollout status statefulset/spark-master --timeout=600s
```



## Wait for spark-worker

```bash
kubectl rollout status statefulset/spark-worker --timeout=600s
```



## Store the Python job

```bash
cat > pyspark/setup_smoke.py <<'PY'
from pyspark.sql import SparkSession
spark = (SparkSession.builder.appName('workspace_spark_setup')
    .master('spark://spark-master:7077')
    .config('spark.driver.host', 'jupyter.bigdata.svc.cluster.local')
    .config('spark.driver.bindAddress', '0.0.0.0')
    .config('spark.driver.port', '7078')
    .config('spark.blockManager.port', '7079')
    .config('spark.cores.max', '1')
    .config('spark.executor.cores', '1')
    .config('spark.executor.memory', '512m')
    .config('spark.openlineage.disabled', 'true').getOrCreate())
try:
    spark._jvm.org.apache.logging.log4j.LogManager.getLogger('workspace.setup').warn('workspace-spark-driver')
    def calculate(value):
        print('workspace-spark-executor', flush=True)
        return value * 2
    assert spark.sparkContext.parallelize([21], 1).map(calculate).collect() == [42]
    spark.range(3).write.mode('overwrite').parquet('hdfs://namenode:9000/user/team/test-spark')
    assert spark.read.parquet('hdfs://namenode:9000/user/team/test-spark').count() == 3
    print('PASS: distributed Python executor and HDFS Parquet round-trip.')
finally:
    spark.stop()
PY
```

The script stays in your host workspace and is immediately visible to Jupyter.

## Run the distributed job

```bash
kubectl exec jupyter-0 -c jupyter -- python /workspace/pyspark/setup_smoke.py
```



## Check collected logs and metrics

```bash
python3 validations/check-observability.py --components spark-master,spark-worker,jupyter --jobs spark-master,spark-worker
```

The helper makes temporary loopback API forwards, waits for actual log/metric samples and closes its forwards. It fails if collection is missing. Allow several scrape intervals after startup.

## Check driver and executor logs

```bash
python3 validations/check-observability.py --components jupyter --file-logs --marker workspace-spark-driver
python3 validations/check-observability.py --components spark-worker --file-logs --marker workspace-spark-executor
```



## Open the Spark browser

Run this in a second terminal and keep it open. Continue the setup in your original terminal.

```bash
kubectl port-forward --address=127.0.0.1 service/spark-master 8080:8080
```

Use http://127.0.0.1:8080 in the Guacamole browser.

## Stop Spark

```bash
bash stop-components.sh spark
```

Source stays on the host and Parquet output stays in HDFS.

## Resume Spark

```bash
bash start-components.sh spark
```



## Repeat the job

```bash
kubectl exec jupyter-0 -c jupyter -- python /workspace/pyspark/setup_smoke.py
```

